In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# 5.ENCODER（MLP-based）
# [B, N, 128] → [B, N, 128]
# Architecture: Linear(128→256) → BN → GELU → Dropout → Linear(256→128)
class MLPEncoder(nn.Module):
    def __init__(self, d_model: int=128, hidden_dim: int=256, dropout: float=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim), #layer1 :128 -> 256
            nn.BatchNorm1d(hidden_dim), # BatchNorm (applied over flattened)
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model), #layer1 :256 -> 128
        )

    def forward(self, x) -> torch.Tensor:
        # input -> x: [B, N, 128]
        B, N, D = x.shape
        x_flat = x.reshape(B*N, D)
        out = self.net(x_flat)
        return out.reshape(B, N, D) 

"d_model: int=128"中的int主要是为了注释和说明， “-> torch.Tensor”也是同样的目的  

必须使用permute 的前提条件是：当尝试把一个 3D 张量(B, N, D)直接传给 BatchNorm1d 时  
x = x.permute(0, 2, 1)  
x = self.batchnorm1d(x)  
x = x.permute(0, 2, 1)  

为了更加简洁写法，基于Linear层只改变最后一维的维度，而与最后一维前的维度大小无关，与输入是几D张量无关，因此：  
“x_flat = x.reshape(B*N, D)” 将3D压缩成2D，例如：[32, 8, 128] -> [256, 128]  
“32 个样本，每个样本 8 个片段” -> “直接扔过来了 256 个互相独立的样本”  
BatchNorm1d只会认为维度为1的是特征（2D是128， 3D就是8了，3D不转换就会出错），并进行计算  

In [ ]:
# 6. POSITIONAL ENCODING
#    Sinusoidal positional encoding added to encoder output.
#    位置编码中的max_len对应的值是序列长度只有8，
#    这里序列长度是由NLP的Transformer中：“句子单词个数代表句子的长度（I have a pen 4个单词，序列长度为4）”引申的
#    论文中，单个信号样本长度为256，划分为8个长度为32的Patch（片段）
#    因此信号是时间序列，前后存在物理联系，而Transformer并不会判断句子中单词的前后顺序，因此需要对词的位置进行信息编码
#    这里采用的Sinusoidal PE 正弦/余弦编码， 具有绝对位置的唯一性和相对距离的可计算性，相对距离的可计算性提供了位置的时间信息
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int=128, max_len: int=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe) # [1, max_len, d_model]

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        """x: [B, N, D] → [B, N, D] + positional info"""
        x = x + self.pe[:, :x.size(1), :]
        return x

In [ ]:
# 7. TRANSFORMER MODULE
#    2-layer Transformer Encoder; 8-head attention; d_model=128
class TransformerModule(nn.Module):
    """
    2 * TransformerEncoderLayer:
      Multi-Head Self-Attention (8 heads) → FFN → LayerNorm → Residual
    Output: [B, N, 128]
    """
    def __init__(self, d_model: int = 128, nhead: int=8,
                 dim_feedforward: int = 512, dropout: float = 0.1, num_layers: int = 2):
        super.__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=False, # Post-LN (standard)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      """x: [B, N, 128] → [B, N, 128]"""
      return self.transformer(x)

默认情况 (batch_first=False)： Transformer 期望的输入形状是 [Sequence, Batch, Feature]  
(batch_first=True): [Batch, Sequence, Feature]

---

### 2. `norm_first`：归一化应该在“干活前”还是“干活后”？ (架构设计的开关)

这个参数决定了 **Layer Normalization (层归一化 LN)** 在 Transformer 内部模块中的安放位置。它虽然只变动了一个位置，但对模型能不能顺利训练起来有着**天壤之别**的影响。

我们知道，Transformer 层里面有两个主要模块：多头注意力（Self-Attention）和前馈网络（FFN）。同时每个模块都配有残差连接（Residual add）。

#### 方式 A：`norm_first=False` （Post-LN，PyTorch 的默认值）
这被称为 **Post-LN (后置归一化)**。这是 2017 年原版《Attention is All You Need》论文里画的经典结构。
*   **数据流向：** 输入 $x \rightarrow$ 模块干活(Attention/FFN) $\rightarrow$ 把干活的结果和 $x$ 加起来(残差) $\rightarrow$ **做 LayerNorm**。
*   **公式体现：** $x_{\text{out}} = \text{LayerNorm}(x + \text{SubLayer}(x))$
*   **特点与缺点：** 在残差相加**之后**做归一化。这种结构在理论上效果很好，但是**非常难训练！** 很容易在训练初期梯度爆炸。为了让它稳定收敛，通常必须配合一种叫做“Learning Rate Warmup（学习率预热）”的复杂技巧（也就是一开始学习率极小，慢慢变大，再慢慢变小）。

#### 方式 B：`norm_first=True` （Pre-LN，现代大模型的首选）
这被称为 **Pre-LN (前置归一化)**。这是后来学者改进的结构，现在几乎所有的超级大模型（包括 GPT-3, BERT 的各种变体, LLaMA 等）全部采用这种结构。
*   **数据流向：** 输入 $x \rightarrow$ **先做 LayerNorm** $\rightarrow$ 模块干活(Attention/FFN) $\rightarrow$ 把干活的结果和最初始的 $x$ 加起来(残差)。
*   **公式体现：** $x_{\text{out}} = x + \text{SubLayer}(\text{LayerNorm}(x))$
*   **特点与优点：** 在模块干活**之前**做归一化。仔细看它的公式，外层是一个纯净的 $x + (...)$。这意味着反向传播时，有一条“畅通无阻的高速公路”可以让梯度直接流回最底层。**它极大地提高了训练的稳定性！** 通常不需要复杂的学习率预热也能轻松收敛。

In [ ]:
# {存在问题}
# 8. CLASSIFICATION HEAD
#    Attention Pooling → MLP-Classifier → Predicted Labels
#    Also outputs Device Embedding [B, 128] for cosine-similarity open-set detection
class AttentionPooling(nn.Module):
    """Attention-based pooling: learns a query to weight all N patch embeddings.(学习一个查询向量，为所有 N 个分块嵌入分配权重)"""
    def __init__(self, d_model: int = 128):
        super().__init__()

        self.atten = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, N, D] → [B, D]"""
        weights = self.attn(x)                 # [B, N, 1]
        weights = F.softmax(weights, dim=1)    # normalize over patches
        out = (x * weights).sum(dim=1)         # [B, D]
        return out      

① x 的形状是 [Batch, 8, 128]（每个 patch 有 128 个特征）。  
  weights 的形状是 [Batch, 8, 1]。  
  从数学上来讲[batch, 8]等效于[Batch, 8, 1]，但是介于张量计算的广播机制，一般情况下不可省略1  
  比如这里相乘的时候，PyTorch 会自动把那个 1 复制扩展 128 次，第 1 个 patch 算出来的那 1 个 注意力得分，
  会公平地乘到第 1 个 patch 的 128 个 特征数值上，以实现整体缩放  

② 打分： 线性层给 8 个 patch 各自打了一个原始分  
  分配配额 (Softmax)： 把这 8 个分数变成百分比（比如 patch 1 占 80%，patch 2 占 5%...），总和为 100% (即 1.0)  
  加权 (x * weights)： 把原始特征按照这些百分比进行放大或缩小（比如 patch 1 的所有特征乘以 0.8，patch 2 乘以 0.05）  
  求和 (.sum)： 沿着 8 个 patch 的方向，把缩放后的特征加在一起  

③ 为什么最后要求和  
  如果输入[Batch, 8, 128]，则分类头只能处理8个patch,求和之后，形状变成了 [Batch, 128]，无论信号有多长，经过池化后都是128维  
  最后的求和，从概率计算上来说，等同于最终的投票  

存在问题：1.self.atten按照论文来说，有可能不存在偏置项，即可能需要bias=False  
2.输入softmax的变量按照论文来说，是会先采用tanh函数进行处理  

论文对应版本：  
import torch  
import torch.nn as nn  
import torch.nn.functional as F  

class AttentionPooling(nn.Module):  
    """  
    Attention-based pooling strictly following the paper's formulation.  
    Learns a weight vector to aggregate N patch embeddings into a device embedding.  
    """  
    def __init__(self, d_model: int = 128):  
        super().__init__()  
        # 严格遵守公式: w^T，不需要 bias  
        self.w = nn.Linear(d_model, 1, bias=False)  

    def forward(self, x: torch.Tensor) -> torch.Tensor:  
        """  
        x: [Batch, N_p, d_model] → 返回: [Batch, d_model]  
        """  
        # 1. 严格遵守公式: tanh 非线性截断，防止极端片段主导  
        x_tanh = torch.tanh(x)  
        
        # 2. 线性映射打分 [Batch, N_p, 1]  
        scores = self.w(x_tanh)  
        
        # 3. Softmax 归一化注意力权重 [Batch, N_p, 1]  
        alpha = F.softmax(scores, dim=1)  
        
        # 4. 优雅的加权求和: 利用广播机制完成 alpha_i * h_i 然后在 dim=1 塌缩  
        return (x * alpha).sum(dim=1)  


In [ ]:
# {存在问题}
class ClassificationHead(nn.Module):
    def __init__(self, d_model: int = 128, num_classes: int = 20):
        super().__init__()
        self.pool = AttentionPooling(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor):
        """
        x: [B, N, 128]
        returns: logits [B, num_classes], device_embedding [B, 128]
        """
        emb = self.pool(x)     # [B, 128]  ← Device Embedding
        logits = self.mlp(emb) # [B, num_classes]
        return logits, emb

贴近论文描述的分类头：  
import torch  
import torch.nn as nn  

class ClassificationHead(nn.Module):  
    def __init__(self, d_model: int = 128, num_classes: int = 20, dropout: float = 0.3):  
        super().__init__()  
        self.pool = AttentionPooling(d_model) # 假设你已经定义好这个类  

        # 第一层: 128 -> 256  
        self.layer1 = nn.Sequential(  
            nn.Linear(d_model, 256),  
            nn.BatchNorm1d(256),    # B(·)  
            nn.ReLU(),              # σ(·)  
            nn.Dropout(dropout)     # D(·)  
        )  

        # 第二层: 256 -> 128  
        self.layer2 = nn.Sequential(  
            nn.Linear(256, 128),  
            nn.BatchNorm1d(128),    # B(·)  
            nn.ReLU(),              # σ(·)  
            nn.Dropout(dropout)     # D(·)  
        )  

        # 输出层: 128 -> Nclasses  
        self.output_layer = nn.Linear(128, num_classes)  

    def forward(self, x: torch.Tensor):  
        # 1. Attention Pooling  
        emb = self.pool(x) # [B, 128]  
        
        # 2. 论文公式流程  
        h1 = self.layer1(emb) # [B, 256]  
        h2 = self.layer2(h1)  # [B, 128]  
        logits = self.output_layer(h2) # [B, num_classes]  
        
        return logits, emb  

In [ ]:
# {存在问题}
# 9. RECONSTRUCTION HEAD (DECODER)
#    [B, N, 128] → Decode back to original patch features [B, N, 64]
#    Linear(128→256) → BN → GELU → Dropout → Linear(256→128) → Reshape → [B,2,256]
class ReconstructionHead(nn.Module):
    def __init__(self, d_model: int = 128, patch_channels: int = 2, patch_size: int = 32):
        super().__init__()
        out_dim = patch_channels * patch_size # 64
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 256), # Layer 1: 128 → 256
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),     # Layer 2: 256 → 128
            nn.GELU(),
            nn.Linear(128, out_dim), # Project to patch output
        )
        self.patch_channels = patch_channels
        self.patch_size = patch_size

    def forward(self, x):
        """
        x: [B, N, 128]
        returns: reconstructed signal [B, 2, 256]
        """
        B, N, D = x.shape
        x_flat = x.reshape(B * N, D)
        out = self.decoder(x_flat)  # [B*N, 64]
        out = out.reshape(B, N, self.patch_channels, self.patch_size) # [B,8,2,32]
        out = out.permute(0, 2, 1, 3).reshape(B, self.patch_channels, -1)
        return out

版本1 ↑ 数据格式变化相近，但多了一个线性层，结构有区别  

版本2（根据论文中，只有两层线性层，且按照重构升维的理念构建）  
结构相近，数据格式变化有差别  
class Decoder_mlp(nn.Module):  
    def __init__(self, dropout=0.1):  
        super().__init__()  
        self.linear1 = nn.Linear(128, 256)   
        self.batchnorm1d = nn.BatchNorm1d(256)    
        self.gelu = nn.GELU()  
        self.dropout = nn.Dropout(dropout)  
        self.linear2 = nn.Linear(256 * 8, 512)  

    def forward(self, x):  
        x = self.linear1(x)  
        x = x.permute(0, 2, 1)  
        x = self.batchnorm1d(x)  
        x = x.permute(0, 2, 1)  
        x = self.gelu(x)  
        x = self.dropout(x)  

        x = x.flatten(1, 2)   

        x = self.linear2(x)  
        x = x.reshape(-1, 2, 256)  

        return x   

### PyTorch 中 `self.` 到底存了什么？

在继承了 `nn.Module` 的类中，`self` 可以挂载三种不同类型的东西：

1.  **子模块 (Sub-modules) / 可学习参数**:
    *   例如：`self.decoder = nn.Sequential(...)`
    *   PyTorch 的底层的机制会自动扫描到这些子模块，提取出里面的 Weights 和 Biases。当调用 `optimizer.step()` 时，这些会被更新。
2.  **缓冲区 (Buffers)**:
    *   例如：之前代码里的位置编码 `self.register_buffer('pe', pe)`。
    *   它们是张量（Tensor），不是参数不更新梯度，但模型 `to('cuda')` 时它会跟着去 GPU，且会保存在权重文件里。
3.  **普通的 Python 属性 (Python Attributes)**:
    *   例如：`self.patch_channels = patch_channels`（只是一个普通的整数 `int`）。
    *   **PyTorch 根本不关心它们**。它们不会占用显存，不会被保存在模型的权重文件（`state_dict`）里，也不会参与梯度计算。它们的存在，**纯粹是为了当“配置参数”，方便类内部的其他函数调用计算而已。**


**在 Python 中，`__init__` 和 `forward` 是两个完全独立的函数，它们之间有一堵“墙”（作用域隔离）。**

在 `def __init__(self, patch_channels: int = 2, ...):` 括号里定义的 `patch_channels`，仅仅是 `__init__` 这个函数的**局部变量（Local Variable）**。

### 1. 通俗的比喻：两个不同的房间

*   把 `__init__` 想象成**1号房间**（用于准备物资）。
*   把 `forward` 想象成**2号房间**（用于实际干活）。
*   把 `self` 想象成一个**贯穿所有房间的公共储物柜**。

当你在实例化模型时，你把 `patch_channels=2` 递交给了1号房间（`__init__`）。
但是，**当1号房间的任务执行完毕后，里面的所有临时物资（局部变量）都会被立刻销毁扔掉**。

等到模型开始前向传播，进入2号房间（`forward`）时，如果你直接喊 `patch_channels`，2号房间里根本没有这个东西，它已经被销毁了，所以 Python 会报错。

**为了让2号房间也能用到这个数字，你必须在1号房间销毁前，把它存进公共储物柜里**，也就是写下：
`self.patch_channels = patch_channels`。

到了2号房间，你就可以通过打开储物柜（`self.patch_channels`）把它拿出来用了。